In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

DATA_DIR  = '/kaggle/input/datasets/rafisongram/ras-compound-character-dataset/RAS-Compound-character-dataset'
TRAIN_DIR = f"{DATA_DIR}/train"
TEST_DIR  = f"{DATA_DIR}/test"

# Smaller images = much faster on CPU
img_size = (32, 32)
batch_size = 16   # smaller batches = less memory pressure, fine for CPU

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=8,
    width_shift_range=0.04,
    height_shift_range=0.04,
    zoom_range=0.08,
    validation_split=0.15
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=img_size, color_mode='grayscale',
    batch_size=batch_size, class_mode='categorical', subset='training'
)
val_data = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=img_size, color_mode='grayscale',
    batch_size=batch_size, class_mode='categorical', subset='validation', shuffle=False
)
test_data = test_datagen.flow_from_directory(
    TEST_DIR, target_size=img_size, color_mode='grayscale',
    batch_size=batch_size, class_mode='categorical', shuffle=False
)

num_classes = train_data.num_classes
print("Classes:", num_classes)

In [17]:
model = models.Sequential([
    layers.Input(shape=(32, 32, 1)),

    layers.Conv2D(16, 3, padding='same', activation='relu'),  # one regular conv to start
    layers.MaxPooling2D(),

    layers.SeparableConv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),

    layers.SeparableConv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_7 (Conv2D)               │ (None, 32, 32, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 16, 16, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_2              │ (None, 16, 16, 32)     │           688 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 8, 8, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_3              │ (None, 8, 8, 64)       │         2,400 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,538 (29.45 KB)

 Trainable params: 7,538 (29.45 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
]

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=30,          # early stopping will cut this short if it plateaus
    callbacks=callbacks,
    verbose=1
)

Epoch 1/30
417/417 ━━━━━━━━━━━━━━━━━━━━ 17s 36ms/step - accuracy: 0.8949 - loss: 0.3632 - val_accuracy: 0.8962 - val_loss: 0.3341 - learning_rate: 0.0010
Epoch 2/30
417/417 ━━━━━━━━━━━━━━━━━━━━ 14s 34ms/step - accuracy: 0.8958 - loss: 0.3418 - val_accuracy: 0.8962 - val_loss: 0.3374 - learning_rate: 0.0010
Epoch 3/30
417/417 ━━━━━━━━━━━━━━━━━━━━ 15s 35ms/step - accuracy: 0.8958 - loss: 0.3422 - val_accuracy: 0.8962 - val_loss: 0.3340 - learning_rate: 0.0010
Epoch 4/30
417/417 ━━━━━━━━━━━━━━━━━━━━ 15s 36ms/step - accuracy: 0.8958 - loss: 0.3408 - val_accuracy: 0.8962 - val_loss: 0.3337 - learning_rate: 0.0010
Epoch 5/30
417/417 ━━━━━━━━━━━━━━━━━━━━ 15s 35ms/step - accuracy: 0.8958 - loss: 0.3399 - val_accuracy: 0.8962 - val_loss: 0.3368 - learning_rate: 0.0010
Epoch 6/30
417/417 ━━━━━━━━━━━━━━━━━━━━ 14s 34ms/step - accuracy: 0.8958 - loss: 0.3390 - val_accuracy: 0.8962 - val_loss: 0.3415 - learning_rate: 0.0010
Epoch 7/30
417/417 ━━━━━━━━━━━━━━━━━━━━ 15s 36ms/step - accuracy: 0.8958 - l

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

class_names = list(train_data.class_indices.keys())
try:
    mapping_df = pd.read_csv("../ras_class_mapping.csv")
    id_to_char = dict(zip(mapping_df["class_id"].astype(str), mapping_df["bengali_character"]))
    display_names = [id_to_char[c] for c in class_names]
except FileNotFoundError:
    display_names = class_names

y_true = test_data.classes
y_pred = model.predict(test_data, verbose=1).argmax(axis=1)

test_acc = (y_true == y_pred).mean()
cer = 1 - test_acc  # single-character samples: CER == top-1 error rate
print(f"Test Accuracy: {test_acc:.4f}")
print(f"CER          : {cer:.4f}\n")
print(classification_report(y_true, y_pred, target_names=display_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, cmap="Blues", xticklabels=display_names, yticklabels=display_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Confusion Matrix (CER: {cer:.4f})")
plt.tight_layout()
plt.show()